In [3]:
! pip install  tqdm

In [3]:
import os
import shutil
import random
from tqdm import tqdm
from tkinter import filedialog, Tk

def split_dataset():
    # 1. 원본 폴더 선택 (images와 labels가 들어있는 상위 폴더)
    root = Tk()
    root.withdraw()
    print("📂 'images'와 'labels' 폴더가 포함된 상위 폴더를 선택하세요.")
    base_dir = filedialog.askdirectory(title="데이터셋 상위 폴더 선택")
    root.destroy()

    if not base_dir: return

    # 2. 경로 설정
    src_img_dir = os.path.join(base_dir, 'images')
    src_lbl_dir = os.path.join(base_dir, 'labels')
    
    # 새롭게 저장될 경로 (YOLO 표준 구조)
    target_dir = os.path.join(base_dir, 'YOLO_Dataset')
    split_names = ['train', 'val']
    sub_dirs = ['images', 'labels']

    for s in split_names:
        for d in sub_dirs:
            os.makedirs(os.path.join(target_dir, s, d), exist_ok=True)

    # 3. 파일 목록 수집 및 셔플
    img_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(img_files)

    # 4. 분할 지점 계산 (80% 기준)
    split_idx = int(len(img_files) * 0.8)
    train_files = img_files[:split_idx]
    val_files = img_files[split_idx:]

    def copy_files(file_list, split_name):
        print(f"📦 {split_name} 데이터 복사 중...")
        for img_name in tqdm(file_list):
            file_stem = os.path.splitext(img_name)[0]
            lbl_name = file_stem + ".txt"

            # 이미지 복사
            shutil.copy(os.path.join(src_img_dir, img_name), 
                        os.path.join(target_dir, split_name, 'images', img_name))
            
            # 라벨 복사 (존재할 경우만)
            if os.path.exists(os.path.join(src_lbl_dir, lbl_name)):
                shutil.copy(os.path.join(src_lbl_dir, lbl_name), 
                            os.path.join(target_dir, split_name, 'labels', lbl_name))

    # 5. 실행
    copy_files(train_files, 'train')
    copy_files(val_files, 'val')

    # 6. dataset.yaml 자동 생성
    yaml_content = f"""
path: {os.path.abspath(target_dir)}
train: train/images
val: val/images

names:
  0: fire
  1: smoke
"""
    with open(os.path.join(target_dir, 'dataset.yaml'), 'w') as f:
        f.write(yaml_content.strip())

    print(f"\n✨ 분할 완료! 모든 데이터는 '{target_dir}' 폴더에 정리되었습니다.")
    print(f"📄 'dataset.yaml' 파일이 생성되었습니다. 바로 학습에 사용하세요.")

if __name__ == "__main__":
    split_dataset()

📂 'images'와 'labels' 폴더가 포함된 상위 폴더를 선택하세요.
📦 train 데이터 복사 중...


100%|██████████| 2371/2371 [00:11<00:00, 209.15it/s]


📦 val 데이터 복사 중...


100%|██████████| 593/593 [00:03<00:00, 161.00it/s]


✨ 분할 완료! 모든 데이터는 'E:/DATASET/fire_on_road/fire_on_road\YOLO_Dataset' 폴더에 정리되었습니다.
📄 'dataset.yaml' 파일이 생성되었습니다. 바로 학습에 사용하세요.


In [ ]:
# 증강학습


In [ ]:
from ultralytics import YOLO

def train_car_model():
    # 1. 사전 학습된 모델 로드 (가장 가벼운 nano 모델 기준)
    # 인식률을 더 높이고 싶다면 'yolov8s.pt' 또는 'yolov8m.pt'를 사용하세요.
    model = YOLO('best_sb.pt') 

    # 2. 모델 학습 시작
    model.train(

        workers=0,  # 데이터 로딩 시 CPU 프로세스를 0으로 설정 (테스트용)
        device='0',
        data="E://DATASET/fire_on_road/fire_on_road/YOLO_Dataset/dataset.yaml",   # 설정 파일 경로
        epochs=100,            # 학습 횟수 (데이터 양에 따라 조절)
        imgsz=640,             # 이미지 크기
        batch=16,              # 배치 사이즈 (GPU 메모리에 따라 조절)
        patience=20,           # 성능 개선이 없을 때 조기 종료하는 기준
        project='CarDetection',# 결과 저장 프로젝트명
        name='v1_manual_data'  # 상세 실험 이름
        
    )

    print("✅ 학습이 완료되었습니다. 결과는 'CarDetection/v1_manual_data' 폴더에서 확인하세요.")

if __name__ == "__main__":
    train_car_model()

New https://pypi.org/project/ultralytics/8.4.19 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.18  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/DATASET/fire_on_road/fire_on_road/YOLO_Dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best_sb.pt, momentum=0.937, mosa

In [8]:
import torch
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"사용 가능한 GPU 개수: {torch.cuda.device_count()}")

CUDA 사용 가능 여부: False
사용 가능한 GPU 개수: 0


In [15]:
import torch
print(f"현재 환경에서 CUDA 가능? {torch.cuda.is_available()}")

현재 환경에서 CUDA 가능? False
